In [175]:
import numpy as np
import pandas as pd

In [176]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [177]:
df = pd.read_csv('train.csv')

In [178]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [179]:

df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
title_map = {'Mr':'Mr','Mrs':'Mrs','Miss':'Miss','Master':'Master','Ms':'Mrs','Mlle':'Miss','Mme':'Mrs'}
df['Title'] = df['Title'].map(title_map).fillna('Rare')

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['HasCabin'] = df['Cabin'].notna().astype(int)


In [180]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [181]:
# Step 1 -> train/test/split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42,
                                                stratify=df['Survived'])

In [182]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,HasCabin
692,3,male,NaN,0,0,56.4958,S,Mr,1,1,0
481,2,male,NaN,0,0,0.0000,S,Mr,1,1,0
527,1,male,NaN,0,0,221.7792,S,Mr,1,1,1
855,3,female,18.0,0,1,9.3500,S,Mrs,2,0,0
801,2,female,31.0,1,1,26.2500,S,Mrs,3,0,0


In [212]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 692 to 507
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Pclass      712 non-null    int64  
 1   Sex         712 non-null    str    
 2   Age         575 non-null    float64
 3   SibSp       712 non-null    int64  
 4   Parch       712 non-null    int64  
 5   Fare        712 non-null    float64
 6   Embarked    710 non-null    str    
 7   Title       712 non-null    str    
 8   FamilySize  712 non-null    int64  
 9   IsAlone     712 non-null    int64  
 10  HasCabin    712 non-null    int64  
dtypes: float64(2), int64(6), str(3)
memory usage: 72.9 KB


In [183]:
y_train.sample(5)

465    0
856    1
551    0
568    0
193    1
Name: Survived, dtype: int64

In [184]:
# Embarked needs imputation THEN encoding -> nest it in a small sub-pipeline
embarked_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [185]:
# One ColumnTransformer, referencing column NAMES -> no index-shifting bug possible
preprocessor = ColumnTransformer([
    ('age', SimpleImputer(strategy='median'), ['Age']),
    ('embarked', embarked_pipe, ['Embarked']),
    ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Sex', 'Pclass', 'Title']),
], remainder='passthrough')  # passes through Fare, SibSp, Parch, FamilySize, IsAlone, HasCabin


In [186]:
pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('scaling', MinMaxScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7932960893854749


In [192]:
# Tune it
params = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [3, 5, 7, None],
    'clf__min_samples_split': [2, 5, 10]
}
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print(grid.best_score_, grid.best_params_)

0.8287107258938246 {'clf__max_depth': 5, 'clf__min_samples_split': 2, 'clf__n_estimators': 100}


In [193]:
# # imputation transformer
# trf1 = ColumnTransformer([
#     ('impute_age',SimpleImputer(),[2]),
#     ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
# ],remainder='passthrough')


In [194]:
# # one hot encoding
# trf2 = ColumnTransformer([
#     ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])
# ],remainder='passthrough')

In [195]:
# # Scaling
# trf3 = ColumnTransformer([
#     ('scale',MinMaxScaler(),slice(0,10))
# ])

In [196]:
# # Feature selection
# trf4 = SelectKBest(score_func=chi2,k=5)

In [197]:
# # train the model
# trf5 = DecisionTreeClassifier()

# Create Pipeline

In [198]:
# pipe = Pipeline([
#     ('trf1',trf1),
#     ('trf2',trf2),
#     ('trf3',trf3),
#     ('trf4',trf4),
#     ('trf5',trf5)
# ])

# Pipeline Vs make_pipeline

Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

In [199]:
# Alternate Syntax
#pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [200]:
# # train
# pipe.fit(X_train,y_train)

# Explore the Pipeline

In [201]:
# # Code here
# pipe.named_steps

In [202]:
# # to check indiviidual parameters in pipeline
# pipe.named_steps['trf1'].transformers_[1][1].statistics_

In [203]:
# Display Pipeline

from sklearn import set_config
set_config(display='diagram')

In [204]:
# Predict
y_pred = pipe.predict(X_test)

In [205]:
y_pred

array([0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0])

In [206]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7932960893854749

# Cross Validation using Pipeline

In [207]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.7964936471978726)

# Exporting the Pipeline

In [208]:
# export 
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))